# 01 – Data Check

Quick sanity check of the downloaded raw prices and computed log-returns.
Verifies completeness, identifies NaNs, and plots time series.

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from omegaconf import OmegaConf

cfg = OmegaConf.load('../configs/data.yaml')
raw_dir = Path('../') / cfg.raw_dir
processed_dir = Path('../') / cfg.processed_dir

## 1. Raw prices

In [ ]:
prices = pd.read_csv(raw_dir / 'prices.csv', index_col=0, parse_dates=True)
print('Shape:', prices.shape)
print('Date range:', prices.index.min(), '→', prices.index.max())
print('\nMissing values per ticker:')
print(prices.isna().sum())

In [ ]:
fig, axes = plt.subplots(len(prices.columns), 1, figsize=(12, 2.5 * len(prices.columns)), sharex=True)
for ax, col in zip(axes, prices.columns):
    ax.plot(prices.index, prices[col], linewidth=0.8)
    ax.set_ylabel(col)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[-1].set_xlabel('Date')
fig.suptitle('Adjusted Close Prices', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../outputs/figures/prices.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Log-returns

In [ ]:
returns = pd.read_csv(processed_dir / 'returns_raw.csv', index_col=0, parse_dates=True)
print('Shape:', returns.shape)
print('\nDescriptive statistics:')
returns.describe().round(4)

In [ ]:
fig, axes = plt.subplots(1, len(returns.columns), figsize=(4 * len(returns.columns), 4))
for ax, col in zip(axes, returns.columns):
    ax.hist(returns[col].dropna(), bins=80, density=True, alpha=0.7)
    ax.set_title(col)
    ax.set_xlabel('Log-return')
fig.suptitle('Return distributions', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/figures/return_histograms.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Correlation heatmap

In [ ]:
import seaborn as sns
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(returns.corr(), annot=True, fmt='.2f', cmap='RdYlGn', center=0, ax=ax)
ax.set_title('Pearson correlation – log-returns')
plt.tight_layout()
plt.savefig('../outputs/figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()